In [ ]:
import pandas as pd
from rapidfuzz import fuzz
import re

df = pd.read_excel(r"data/Name_Mapping.xlsx", sheet_name="Sheet1")

STOPWORDS = {
    "LIMITED", "LTD", "PRIVATE", "PVT", "PVT LTD", "COMPANY", "CO",
    "CORPORATION", "CORP", "ENTERPRISES", "ENTERPRISE", "INDUSTRIES",
    "INDUSTRY", "TECHNOLOGIES", "SYSTEMS", "SOLUTIONS",
    "SERVICES", "SERVICE", "VENTURES", "GLOBAL", "INTERNATIONAL",
    "GROUP", "ASSOCIATES", "CONSULTANTS", "HOLDINGS", "INVESTMENTS",
    "PARTNERS", "PARTNERSHIP", "NBFC", "FINANCIAL", "CAPITAL", "CREDIT",
    "TRUST", "LIFE", "INSURANCE", "GENERAL", "SECURITIES",
    "NATIONAL", "DEPARTMENT", "COUNCIL", "AUTHORITY", "BOARD",
    "COMMISSION", "UNIVERSITY", "COLLEGE", "INDIAN", "BHARAT"
}

def clean_text(s):
    if pd.isna(s):
        return None
    s = str(s).strip().upper()
    s = re.sub(r"\s+", " ", s)
    s = s.replace("\u00A0", " ")
    s = re.sub(r"[^A-Z0-9&().,\- ]", "", s)
    if s in ["#N/A", "N/A", "NA", "NONE", "NULL", ""]:
        return None
    return s

def strip_stopwords(s):
    if not s:
        return None
    s = str(s).strip().upper()
    tokens = [t for t in s.split() if t not in STOPWORDS]
    return " ".join(tokens) if tokens else s

def hybrid_score(query, choice):
    GENERIC_PREFIXES = {
        "TATA", "ADITYA", "RELIANCE", "HDFC", "ICICI", "BAJAJ", "MAHINDRA",
        "ADANI", "LARSEN", "L&T", "INFOSYS", "WIPRO", "HERO", "JSW", "BIRLA",
        "INDIA", "INDIAN", "BHARAT", "NATIONAL", "STATE", "BANK"
    }

    GENERIC_WORDS = STOPWORDS

    if not query or not choice:
        return 0

    query_tokens = query.split()
    choice_tokens = choice.split()
    query_first = query_tokens[0] if query_tokens else ""
    choice_first = choice_tokens[0] if choice_tokens else ""

    score = (
        0.5 * fuzz.token_set_ratio(query, choice)
        + 0.3 * fuzz.partial_ratio(query, choice)
        + 0.2 * fuzz.token_sort_ratio(query, choice)
    )

    core_query = [t for t in query_tokens if t not in GENERIC_WORDS]
    core_choice = [t for t in choice_tokens if t not in GENERIC_WORDS]
    core_query_str = " ".join(core_query)
    core_choice_str = " ".join(core_choice)
    core_score = fuzz.token_set_ratio(core_query_str, core_choice_str) if core_query and core_choice else 0

    set_q = set(core_query)
    set_c = set(core_choice)
    jaccard = len(set_q & set_c) / len(set_q | set_c) if (set_q | set_c) else 0

    if (
        query_first == choice_first
        and query_first in GENERIC_PREFIXES
        and query_first != ""
    ):
        subset_flag = set_q.issubset(set_c) or set_c.issubset(set_q)
        if jaccard < 0.5 or subset_flag or core_score < 70:
            score -= 80

    if core_score < 60 and score > 85:
        score -= 30

    if query in choice or choice in query:
        score += 10

    return min(100, max(0, score))

def find_matches_per_column(main_name, choices_dict, thresholds=None):
    if main_name is None:
        return {col: (None, None, "No", "No Match") for col in choices_dict.keys()}

    if thresholds is None:
        thresholds = {col: 85 for col in choices_dict.keys()}

    main_stripped = strip_stopwords(main_name)
    col_matches = {}

    for col, choices in choices_dict.items():
        stripped_choices = [strip_stopwords(c) for c in choices]
        scores = [hybrid_score(main_stripped, c) for c in stripped_choices]

        if not scores:
            col_matches[col] = (None, None, "No", "No Match")
            continue

        best_idx = max(range(len(scores)), key=lambda i: scores[i])
        score = scores[best_idx]
        original_choice = choices[best_idx]

        exact_flag = "Yes" if str(main_name).strip().upper() == str(original_choice).strip().upper() else "No"
        if exact_flag == "Yes":
            score = 100
            note = "Exact Match"
        elif score >= thresholds[col]:
            note = "High Confidence"
        elif score >= thresholds[col] * 0.7:
            note = "Medium Confidence"
        else:
            note = "Low Confidence"

        if score >= thresholds[col]:
            col_matches[col] = (original_choice, score, exact_flag, note)
        else:
            col_matches[col] = (None, score, exact_flag, note)

    return col_matches

for col in df.columns:
    df[col] = df[col].map(clean_text)

choices_dict = {
    "Reference_1": df["Reference_1"].dropna().tolist(),
    "Reference_2": df["Reference_2"].dropna().tolist(),
    "Reference_3": df["Reference_3"].dropna().tolist(),
    "Reference_4": df["Reference_4"].dropna().tolist()
}

thresholds = {col: 90 for col in choices_dict.keys()}

results = []
match_count = 0

for name in df["Main_Names"]:
    matches = find_matches_per_column(name, choices_dict, thresholds)
    row = {
        "Main_Name": name,
        "Ref1_Match": matches["Reference_1"][0],
        "Ref1_Score": matches["Reference_1"][1],
        "Ref1_Exact": matches["Reference_1"][2],
        "Ref1_Note": matches["Reference_1"][3],
        "Ref2_Match": matches["Reference_2"][0],
        "Ref2_Score": matches["Reference_2"][1],
        "Ref2_Exact": matches["Reference_2"][2],
        "Ref2_Note": matches["Reference_2"][3],
        "Ref3_Match": matches["Reference_3"][0],
        "Ref3_Score": matches["Reference_3"][1],
        "Ref3_Exact": matches["Reference_3"][2],
        "Ref3_Note": matches["Reference_3"][3],
        "Ref4_Match": matches["Reference_4"][0],
        "Ref4_Score": matches["Reference_4"][1],
        "Ref4_Exact": matches["Reference_4"][2],
        "Ref4_Note": matches["Reference_4"][3],
    }

    if any([row["Ref1_Match"], row["Ref2_Match"], row["Ref3_Match"], row["Ref4_Match"]]):
        match_count += 1

    results.append(row)

results_df = pd.DataFrame(results)
output_path = r"output/matched_results.xlsx"
results_df.to_excel(output_path, index=False)

print(f"Matching complete. Results saved to {output_path}")
print(f"Total Main Names matched: {match_count} / {len(df['Main_Names'])}")
